# Football-LLM · Notebook 3b · CoT Re-run

**Why this exists:** the first CoT pass (in `03_icl_cot_baselines.ipynb`) had two bugs:
1. `max_new_tokens=512` wasn't enough for the 5-step reasoning — the model ran out of tokens before emitting a `Prediction:` / `Score:` line.
2. `raw_output` was truncated to 800 chars, cutting off any final answer that *did* emit.

The parser then fell back to bare `N-M` matching inside reasoning text, producing nonsense scores (e.g. `303-167` from squad-goal totals).

**This notebook does a minimal, surgical re-run:**
- `max_new_tokens = 1024`
- `raw_output` stored in full (no truncation)
- Parser tightened: bare `N-M` fallback only counts if the numbers are plausibly a score (both ≤ 15).

**Does not re-run ICL.** Those results are good and already saved in `icl_results.json` / `baseline_metrics.json`.


## 1. Setup

In [2]:
%pip install -q transformers accelerate bitsandbytes peft datasets

from huggingface_hub import notebook_login
notebook_login()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.6 MB/s eta 0:00:00


In [3]:
import torch, json, re, numpy as np
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
print(f"torch {torch.__version__} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")

torch 2.10.0+cu128 | GPU: Tesla T4


## 2. Load eval data

In [4]:
!git clone https://github.com/zanwenfu/football-llm.git 2>/dev/null || true

eval_dataset = load_dataset(
    "json", data_files="football-llm/data/training/eval.jsonl", split="train"
)
print(f"Eval samples: {len(eval_dataset)}")

named_eval, anon_eval = [], []
for s in eval_dataset:
    u = s['messages'][1]['content']
    if 'Team A' in u and 'Team B' in u:
        anon_eval.append(s)
    else:
        named_eval.append(s)
print(f"  named: {len(named_eval)}, anon: {len(anon_eval)}")

Generating train split: 0 examples [00:00, ? examples/s]

Eval samples: 128
  named: 64, anon: 64


## 3. Parser (tightened for CoT)

The key change: the bare-dash score fallback now only matches if both numbers are ≤ 15 (realistic soccer score range). This prevents squad-goal totals like `303-167` from being mistaken for a match score.


In [5]:
def parse_prediction(text):
    """Tightened CoT parser. Same as eval_harness.ipynb but with a realistic-score
    sanity check on the bare N-M fallback."""
    result = None
    home_goals = None
    away_goals = None
    text_lower = text.lower()

    # Parse 'Prediction:' line
    pred_match = re.search(r'prediction[:\s]+(.*?)(?:\n|$)', text_lower)
    if pred_match:
        pt = pred_match.group(1).strip()
        if 'draw' in pt: result = 'draw'
        elif 'home' in pt or 'team a' in pt: result = 'home_win'
        elif 'away' in pt or 'team b' in pt: result = 'away_win'
        elif 'win' in pt: result = 'has_winner'

    # Explicit 'Score:' always wins
    m = re.search(r'score[:\s]+.*?(\d+)\s*[-\u2013]\s*(\d+)', text, re.IGNORECASE)
    if m:
        home_goals, away_goals = int(m.group(1)), int(m.group(2))
    else:
        # Bare-dash fallback — but only if BOTH numbers look like a plausible score.
        # Scan the LAST 400 chars (where conclusions live) and take the LAST match.
        tail = text[-400:] if len(text) > 400 else text
        candidates = list(re.finditer(r'(?<![\d.])(\d+)\s*[-\u2013]\s*(\d+)(?![\d.])', tail))
        for cand in reversed(candidates):
            h, a = int(cand.group(1)), int(cand.group(2))
            if h <= 15 and a <= 15:
                home_goals, away_goals = h, a
                break

    # Override result from score when present
    if home_goals is not None and away_goals is not None:
        if home_goals > away_goals: result = 'home_win'
        elif home_goals < away_goals: result = 'away_win'
        else: result = 'draw'

    return {
        'result': result,
        'home_goals': home_goals,
        'away_goals': away_goals,
        'parsed': result is not None,
    }


def parse_ground_truth(assistant_msg):
    return parse_prediction(assistant_msg)


def compute_metrics(results):
    total = len(results)
    parsed = sum(1 for r in results if r['pred']['parsed'])
    correct_result, correct_score = 0, 0
    h_err, a_err = [], []
    for r in results:
        if not r['pred']['parsed'] or not r['gt']['parsed']:
            continue
        if r['pred']['result'] == r['gt']['result']:
            correct_result += 1
        if (r['pred']['home_goals'] == r['gt']['home_goals']
                and r['pred']['away_goals'] == r['gt']['away_goals']):
            correct_score += 1
        if r['pred']['home_goals'] is not None and r['gt']['home_goals'] is not None:
            h_err.append(abs(r['pred']['home_goals'] - r['gt']['home_goals']))
            a_err.append(abs(r['pred']['away_goals'] - r['gt']['away_goals']))
    return {
        'total': total, 'parsed': parsed,
        'parse_rate': parsed / total if total else 0,
        'result_accuracy': correct_result / parsed if parsed else 0,
        'score_exact_match': correct_score / parsed if parsed else 0,
        'goal_mae': float(np.mean(h_err + a_err)) if h_err else float('nan'),
    }

print("Tightened parser ready.")

Tightened parser ready.


## 4. Load base model

In [6]:
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
model.eval()
print(f"Loaded. GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Loaded. GPU: 5.70 GB


## 5. Run CoT with `max_new_tokens=1024` and full raw_output

**Changes from previous run:**
- `max_new_tokens`: 512 → 1024
- Input truncation: `max_length=1024` → 1536 (scaffold is long)
- Raw output: stored in full, not `[:800]`


In [7]:
COT_SCAFFOLD = (
    "\n\nBefore your final answer, reason step-by-step:\n"
    "Step 1: Compare the two teams' squad goal output. Which is higher and by how much?\n"
    "Step 2: Compare the quality of each team's top scorer.\n"
    "Step 3: Compare defensive stats (tackles, clean sheets, defensive rating).\n"
    "Step 4: Identify any conflicts between these signals and weigh them.\n"
    "Step 5: Based on the above, predict the outcome.\n\n"
    "After your reasoning, conclude in EXACTLY this format on separate lines:\n"
    "Prediction: <home_win|away_win|draw>\n"
    "Score: <home_goals>-<away_goals>\n"
    "Reasoning: <one sentence summary>\n"
    "Keep your step-by-step reasoning concise (2-3 sentences per step) so you have room for the conclusion."
)


def build_cot_messages(eval_sample):
    system_msg = {
        "role": "system",
        "content": eval_sample['messages'][0]['content'] + COT_SCAFFOLD,
    }
    return [system_msg, eval_sample['messages'][1]]


def run_cot(samples, model, tokenizer, model_name="CoT"):
    results = []
    model.eval()
    for i, sample in enumerate(samples):
        is_anon = ('Team A' in sample['messages'][1]['content']
                   and 'Team B' in sample['messages'][1]['content'])
        messages = build_cot_messages(sample)
        gt = parse_ground_truth(sample['messages'][2]['content'])

        input_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(
            input_text, return_tensors="pt", truncation=True, max_length=1536
        ).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=1024,      # was 512
                temperature=0.1,
                top_p=0.9,
                do_sample=True,
                repetition_penalty=1.1,
            )
        response = tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
        )
        pred = parse_prediction(response)
        results.append({
            'sample_idx': i,
            'is_anon': is_anon,
            'gt': gt,
            'pred': pred,
            'raw_output': response,     # FULL output, no truncation
            'output_len': len(response),
        })
        if (i + 1) % 10 == 0:
            print(f"  [{model_name}] {i+1}/{len(samples)} done")
    return results


print("Running CoT (fixed) on full eval set (128 samples)...")
cot_results = run_cot(eval_dataset, model, tokenizer, "CoT")
with open("cot_results_v2.json", "w") as f:
    json.dump(cot_results, f, indent=2, default=str)
print(f"Saved cot_results_v2.json ({len(cot_results)} preds)")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Running CoT (fixed) on full eval set (128 samples)...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  [CoT] 10/128 done


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

  [CoT] 110/128 done


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  [CoT] 120/128 done


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Saved cot_results_v2.json (128 preds)


## 6. Validate that the rerun is sane

Three checks before we trust these numbers:
1. Do the outputs reach a final `Prediction:` line?
2. Do all predicted scores look realistic (both ≤ 15)?
3. Does the rerun produce qualitatively different (non-fallback-driven) results?


In [8]:
import numpy as np, re

n_with_prediction = sum(1 for r in cot_results if re.search(r'prediction[:\s]', r['raw_output'].lower()))
n_with_score_line = sum(1 for r in cot_results if re.search(r'score[:\s]', r['raw_output'].lower()))
n_with_step5 = sum(1 for r in cot_results if 'step 5' in r['raw_output'].lower())
output_lens = [r['output_len'] for r in cot_results]
print(f"Output length: min={min(output_lens)}, max={max(output_lens)}, mean={np.mean(output_lens):.0f}")
print(f"Outputs containing 'Prediction:' : {n_with_prediction}/128")
print(f"Outputs containing 'Score:'      : {n_with_score_line}/128")
print(f"Outputs reaching 'Step 5'        : {n_with_step5}/128")

# Sanity: any absurd scores?
absurd = [r for r in cot_results if r['pred']['home_goals'] is not None and
          (r['pred']['home_goals'] > 15 or (r['pred']['away_goals'] or 0) > 15)]
print(f"\nAbsurd predictions (goal > 15): {len(absurd)}")

# Parse rate
parsed = sum(1 for r in cot_results if r['pred']['parsed'])
print(f"Parse rate: {parsed}/128 = {parsed/128*100:.1f}%")

Output length: min=761, max=1399, mean=1024
Outputs containing 'Prediction:' : 128/128
Outputs containing 'Score:'      : 128/128
Outputs reaching 'Step 5'        : 125/128

Absurd predictions (goal > 15): 0
Parse rate: 128/128 = 100.0%


## 7. Metrics (CoT only)

In [9]:
def split_metrics(results):
    named = [r for r in results if not r['is_anon']]
    anon  = [r for r in results if r['is_anon']]
    return {
        'overall': compute_metrics(results),
        'named':   compute_metrics(named),
        'anon':    compute_metrics(anon),
    }

cot_metrics_v2 = split_metrics(cot_results)

print("=" * 80)
print(f"{'Condition':<25} {'Parse%':>8} {'Result Acc':>12} {'Score EM':>10} {'Goal MAE':>10}")
print("=" * 80)
for name, m in [
    ("CoT overall (v2, fixed)", cot_metrics_v2['overall']),
    ("  CoT named (v2)",         cot_metrics_v2['named']),
    ("  CoT anon  (v2)",         cot_metrics_v2['anon']),
]:
    acc = f"{m['result_accuracy']*100:.1f}%"
    sc  = f"{m['score_exact_match']*100:.1f}%"
    mae = f"{m['goal_mae']:.2f}" if not np.isnan(m['goal_mae']) else "N/A"
    pr  = f"{m['parse_rate']*100:.1f}%"
    print(f"{name:<25} {pr:>8} {acc:>12} {sc:>10} {mae:>10}")
print("=" * 80)

# Merge with existing ICL metrics from baseline_metrics.json
import os
if os.path.exists("baseline_metrics.json"):
    with open("baseline_metrics.json") as f:
        bl = json.load(f)
    bl['cot'] = cot_metrics_v2
    with open("baseline_metrics.json", "w") as f:
        json.dump(bl, f, indent=2, default=str)
    print("\nUpdated baseline_metrics.json with fixed CoT results.")
else:
    with open("baseline_metrics.json", "w") as f:
        json.dump({'cot': cot_metrics_v2}, f, indent=2, default=str)
    print("\nWrote baseline_metrics.json (CoT only — no prior ICL found).")

Condition                   Parse%   Result Acc   Score EM   Goal MAE
CoT overall (v2, fixed)     100.0%        44.5%       8.6%       1.09
  CoT named (v2)            100.0%        43.8%       6.2%       1.10
  CoT anon  (v2)            100.0%        45.3%      10.9%       1.08

Wrote baseline_metrics.json (CoT only — no prior ICL found).


## 8. Compare against ICL and FT

Plug into the final table:

| Condition | Named | Anon | Diff (anon − named) |
|---|:---:|:---:|:---:|
| Base Llama (overall, prior run) | — | — | 45.3% overall |
| **ICL (v1)** | 50.0% | 48.4% | −1.6 |
| **CoT (v2, fixed)** | __ | __ | __ |
| **FT (QLoRA, n=384)** | 50.0% | 54.7% | **+4.7** |
| Always home win | 45.3% | 45.3% | 0.0 |

If CoT-v2 still shows anon ≤ named (like ICL did), the paper's story is intact: **only fine-tuning flips the sign of the named/anon gap, consistent with a de-biasing mechanism that requires parameter updates**. If CoT unexpectedly shows anon > named, we'd need to nuance — CoT reasoning may itself act as a partial de-biasing mechanism.
